# Reciter TTS — F5-TTS (русский) → ONNX → Android

F5-TTS — **не-авторегрессионная** flow-matching модель: DiT интегрирует ODE за NFE шагов в мел-спектрограмму, Vocos превращает мел в звук 24 кГц. Нет генерации «токен за токеном», поэтому она в разы быстрее AR-моделей (Qwen3/CosyVoice) на телефоне.

**Важно:** базовый F5 — EN/ZH. Для русского нужен **русский файнтюн + его vocab.txt**. В Ячейке 2 укажи репозиторий/файлы.

На выходе — ZIP для импорта в приложении (Модели → локальный ZIP).


## Ячейка 1: Зависимости


In [ ]:
!pip install -q f5-tts onnx onnxruntime soundfile huggingface_hub
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121 || pip install -q torch torchaudio
import torch, torchaudio, numpy as np, os, json, math, zipfile
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## Ячейка 2: Конфигурация (укажи русский чекпойнт)

Примеры русских F5-файнтюнов на HuggingFace (проверь актуальность и лицензию):
- `Misha24-10/F5-TTS_RUSSIAN`  (файлы вида `*.safetensors` + `vocab.txt`)
- `hotstone228/F5-TTS-Russian`

Поставь `HF_REPO`, `CKPT_FILE`, `VOCAB_FILE_HF`. Либо загрузи свои файлы в `/content` и пропиши пути в `MODEL_CKPT`/`VOCAB_FILE`.


In [ ]:
OUT = "/content/f5-onnx/android"; os.makedirs(OUT, exist_ok=True)
OPSET = 17

# --- Русский чекпойнт ---
HF_REPO       = "Misha24-10/F5-TTS_RUSSIAN"   # репозиторий на HuggingFace (поправь при необходимости)
CKPT_FILE     = "F5TTS_v1_Base_v2/model_last_inference.safetensors"  # имя файла модели в репо
VOCAB_FILE_HF = "F5TTS_v1_Base/vocab.txt"     # имя файла словаря в репо
MODEL_CKPT    = ""   # ЛИБО локальный путь к .safetensors (тогда HF_REPO игнорируется)
VOCAB_FILE    = ""   # ЛИБО локальный путь к vocab.txt

# --- Референсный голос (zero-shot): аудио + ТОЧНЫЙ транскрипт ---
REF_AUDIO = "/content/ref_ru.wav"
REF_TEXT  = "Привет, это проверка синтеза речи."
# Если нет своего — скачаем короткий русский семпл ниже.

# --- Параметры мел/инференса (F5 v1 + Vocos) ---
SR, N_MELS, N_FFT, HOP, WIN = 24000, 100, 1024, 256, 1024
NFE  = 16      # шагов ODE: 8=быстрее, 32=качественнее
CFG  = 2.0     # classifier-free guidance
SWAY = -1.0
SPEED = 1.0
print("OUT =", OUT)

## Ячейка 3: Скачать чекпойнт + (при необходимости) референс


In [ ]:
from huggingface_hub import hf_hub_download

if MODEL_CKPT:
    ckpt_path = MODEL_CKPT
else:
    ckpt_path = hf_hub_download(HF_REPO, CKPT_FILE)
print("ckpt:", ckpt_path)

if VOCAB_FILE:
    vocab_path = VOCAB_FILE
else:
    try:
        vocab_path = hf_hub_download(HF_REPO, VOCAB_FILE_HF)
    except Exception as e:
        from importlib.resources import files
        vocab_path = str(files("f5_tts").joinpath("infer/examples/vocab.txt"))
        print("vocab из репо не найден, беру базовый:", e)
print("vocab:", vocab_path)

# короткий русский референс, если своего нет
if not os.path.exists(REF_AUDIO):
    try:
        from datasets import load_dataset
        ds = load_dataset("google/fleurs", "ru_ru", split="validation", streaming=True, trust_remote_code=True)
        ex = next(iter(ds))
        import soundfile as sf
        a = ex["audio"]["array"]; srr = ex["audio"]["sampling_rate"]
        import librosa
        if srr != SR: a = librosa.resample(np.asarray(a, dtype=np.float32), orig_sr=srr, target_sr=SR)
        sf.write(REF_AUDIO, a[: SR*8], SR)
        REF_TEXT = ex["transcription"]
        print("референс из FLEURS:", REF_TEXT[:60])
    except Exception as e:
        print("не удалось взять референс автоматически — загрузи свой в", REF_AUDIO, "| причина:", e)

## Ячейка 4: Загрузка модели F5 + вокодера


In [ ]:
from f5_tts.infer.utils_infer import load_model, load_vocoder
from f5_tts.model import DiT

model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
model = load_model(DiT, model_cfg, ckpt_path, vocab_file=vocab_path).eval()
vocoder = load_vocoder().eval()

vocab_char_map = {}
with open(vocab_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        vocab_char_map[line[:-1]] = i      # срезаем только \n (пробел — валидный токен!)
print("модель загружена, токенов:", len(vocab_char_map))

## Ячейка 5: Экспорт DiT (с CFG внутри графа) + Vocos, INT8


In [ ]:
import torch.nn as nn

class DiTGuided(nn.Module):
    """Один шаг ODE: гоняет трансформер дважды (cond + полностью drop) и
    возвращает CFG-guided velocity — один onnx-вызов на шаг на устройстве."""
    def __init__(self, t, cfg): super().__init__(); self.t=t; self.cfg=float(cfg)
    def forward(self, x, cond, text, time):
        pred = self.t(x=x, cond=cond, text=text, time=time, drop_audio_cond=False, drop_text=False)
        null = self.t(x=x, cond=cond, text=text, time=time, drop_audio_cond=True,  drop_text=True)
        return null + (pred - null) * self.cfg

transformer = model.transformer if hasattr(model, "transformer") else model
mdit = DiTGuided(transformer, CFG).eval()
T, Lt = 64, 20
xx = torch.randn(1,T,N_MELS); cc = torch.randn(1,T,N_MELS)
tt = torch.randint(0,100,(1,Lt)); tm = torch.rand(1)
with torch.no_grad(): r = mdit(xx,cc,tt,tm)
print("DiT out:", tuple(r.shape))
dit_path = f"{OUT}/f5_dit.onnx"
torch.onnx.export(mdit, (xx,cc,tt,tm), dit_path,
    input_names=["x","cond","text","time"], output_names=["velocity"],
    dynamic_axes={"x":{1:"t"},"cond":{1:"t"},"text":{1:"lt"},"velocity":{1:"t"}},
    opset_version=OPSET, do_constant_folding=True)
print("f5_dit.onnx:", round(os.path.getsize(dit_path)/1024**2), "MB")

class VocosWrap(nn.Module):
    def __init__(self, v): super().__init__(); self.v=v
    def forward(self, mel): return self.v.decode(mel)
mvoc = VocosWrap(vocoder).eval()
mel = torch.randn(1,N_MELS,64)
with torch.no_grad(): rv = mvoc(mel)
print("vocos out:", tuple(rv.shape))
voc_path = f"{OUT}/f5_vocos.onnx"
torch.onnx.export(mvoc, (mel,), voc_path, input_names=["mel"], output_names=["waveform"],
    dynamic_axes={"mel":{2:"t"},"waveform":{1:"n"}}, opset_version=OPSET, do_constant_folding=True)
print("f5_vocos.onnx:", round(os.path.getsize(voc_path)/1024**2), "MB")

# INT8
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    for p in (dit_path, voc_path):
        tmp = p+".int8.onnx"
        quantize_dynamic(p, tmp, weight_type=QuantType.QInt8, per_channel=True)
        if os.path.getsize(tmp) < os.path.getsize(p):
            os.replace(tmp, p); print("quantized", os.path.basename(p), round(os.path.getsize(p)/1024**2), "MB")
        elif os.path.exists(tmp): os.remove(tmp)
except Exception as e:
    print("quant skip:", e)

## Ячейка 6: Словарь + запекание референс-голоса + конфиг + манифест


In [ ]:
def mel_spectrogram(wav):
    ms = torchaudio.transforms.MelSpectrogram(
        sample_rate=SR, n_fft=N_FFT, win_length=WIN, hop_length=HOP, n_mels=N_MELS,
        power=1, center=True, normalized=False, norm=None, mel_scale="htk", f_min=0, f_max=None)
    return torch.log(torch.clamp(ms(wav), min=1e-5))

# словарь
json.dump(vocab_char_map, open(f"{OUT}/vocab.json","w"), ensure_ascii=False)

# запекаем голос(а): (id, locale, display, wav, ref_text)
REF_VOICES = [("ru_ref_1", "ru-RU", "Русский", REF_AUDIO, REF_TEXT)]
metas, blobs, offset = [], [], 0
for vid, locale, disp, wav_path, ref_text in REF_VOICES:
    wav, srr = torchaudio.load(wav_path)
    if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
    if srr != SR: wav = torchaudio.functional.resample(wav, srr, SR)
    m = mel_spectrogram(wav)[0]                 # [100,T]
    Tf = m.shape[1]
    arr = m.t().contiguous().to(torch.float16).cpu().numpy()  # [T,100]
    blobs.append(arr.reshape(-1)); 
    metas.append({"id":vid,"locale":locale,"displayName":disp,"ref_text":ref_text,
                  "n_frames":int(Tf),"offset":int(offset)})
    offset += Tf*N_MELS
    print("baked", vid, "frames", Tf)
np.concatenate(blobs).astype(np.float16).tofile(f"{OUT}/f5_voices.bin")
json.dump({"voices":metas,"dim":N_MELS}, open(f"{OUT}/f5_voices.json","w"), ensure_ascii=False, indent=2)

json.dump({"sample_rate":SR,"n_mels":N_MELS,"n_fft":N_FFT,"hop":HOP,"win":WIN,
           "nfe":NFE,"cfg_strength":CFG,"sway_coef":SWAY,"speed":SPEED},
          open(f"{OUT}/f5_config.json","w"), indent=2)

manifest = {"schemaVersion":1,"id":"f5-tts-ru","displayName":"F5-TTS (Russian)",
  "family":"f5","architecture":"f5","sampleRateHz":SR,
  "tokenizer":{"type":"f5-char","files":["vocab.json","f5_config.json","f5_voices.json","f5_voices.bin"]},
  "files":[{"filename":"f5_dit.onnx","role":"TALKER","sizeMb":round(os.path.getsize(dit_path)/1024**2),"required":True},
           {"filename":"f5_vocos.onnx","role":"VOCODER","sizeMb":round(os.path.getsize(voc_path)/1024**2),"required":True}],
  "voices":[{"id":m["id"],"locale":m["locale"],"displayName":m["displayName"],"speakerId":i} for i,m in enumerate(metas)]}
json.dump(manifest, open(f"{OUT}/model.json","w"), ensure_ascii=False, indent=2)
print("конфиг + манифест записаны")

## Ячейка 7: Сверка — синтез «Привет» эталонной библиотекой
Послушай `f5_ref_privet.wav`: если эталон звучит правильно — значит чекпойнт/референс/словарь корректны, и можно заливать на телефон.


In [ ]:
try:
    from f5_tts.infer.utils_infer import infer_process, preprocess_ref_audio_text
    ra, rt = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)
    wav_ref, sr_lib, _ = infer_process(ra, rt, "Привет, как дела?", model, vocoder,
                                       nfe_step=NFE, cfg_strength=CFG, speed=SPEED)
    import soundfile as sf
    sf.write("/content/f5_ref_privet.wav", wav_ref, sr_lib)
    from IPython.display import Audio, display
    print("длительность:", round(len(wav_ref)/sr_lib,2), "с")
    display(Audio(wav_ref, rate=sr_lib))
except Exception as e:
    print("эталонный инференс не удался:", e)

## Ячейка 8: Сборка архива + загрузка


In [ ]:
zip_path = "/content/f5-tts-ru-android.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, f), f"models/{f}")
print("архив:", round(os.path.getsize(zip_path)/1024**2), "MB")
for f in sorted(os.listdir(OUT)):
    print(" ", f, round(os.path.getsize(os.path.join(OUT,f))/1024**2,2), "MB")
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass